# 03 — Modeling: Dual-Track LightGBM

Trains both tracks via `src.model.train_track` — the same function `python -m src.model` calls — and reports per-class metrics via `src.evaluate.per_class_report`. `RANDOM_STATE = 42` throughout, so this reproduces `runs/model_{full,testbed}.json` exactly.

In [1]:
import pandas as pd
from src.features import PROCESSED_DIR, TARGET_COL
from src.model import train_track

train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
test = pd.read_parquet(PROCESSED_DIR / 'test.parquet')

results = {track: train_track(train, test, track=track) for track in ('full', 'testbed')}
for track, r in results.items():
    m = r['metrics']
    print(f"[{track:>7}] macro-F1={m['macro_f1']:.4f}  acc={m['accuracy']:.4f}  "
          f"threshold-sensitive={m['threshold_sensitive_frac']:.1%}  "
          f"(strawman macro-F1={m['strawman']['macro_f1']:.4f})")

[   full] macro-F1=0.9988  acc=0.9995  threshold-sensitive=0.0%  (strawman macro-F1=0.2051)
[testbed] macro-F1=0.8967  acc=0.9297  threshold-sensitive=17.4%  (strawman macro-F1=0.2051)


**`full` reaches 0.9988 macro-F1 — this is not the headline result, it's the leakage finding restated in model form.** See `docs/LEAKAGE_FINDING.md`: a model at this score on this data is reverse-engineering the synthetic generator's near-disjoint per-class feature ranges, not detecting abuse. `testbed` (0.8967) exists only to give the decision layer (`04_cost_calibration.ipynb`) a non-degenerate region to operate on.

## Per-class report, both tracks

In [2]:
from src.evaluate import per_class_report

for track, r in results.items():
    class_names = list(r['label_encoder'].classes_)
    pred = r['proba'].argmax(axis=1)
    print(f'--- {track} ---')
    display(per_class_report(r['y_test'], pred, class_names).round(4))

--- full ---


,precision,recall,f1-score,support
Fraudulent Return,0.9982,0.9991,0.9987,1112.0000
Legitimate,1.0000,1.0000,1.0000,8345.0000
Policy Abuser,0.9993,0.9965,0.9979,1414.0000
Wardrobing,0.9973,1.0000,0.9987,1129.0000
accuracy,0.9995,0.9995,0.9995,0.9995
macro avg,0.9987,0.9989,0.9988,12000.0000
weighted avg,0.9995,0.9995,0.9995,12000.0000


--- testbed ---


,precision,recall,f1-score,support
Fraudulent Return,0.9917,0.9676,0.9795,1112.0000
Legitimate,0.9759,0.9409,0.9581,8345.0000
Policy Abuser,0.7775,0.8600,0.8167,1414.0000
Wardrobing,0.7762,0.8973,0.8324,1129.0000
accuracy,0.9298,0.9298,0.9298,0.9298
macro avg,0.8803,0.9164,0.8967,12000.0000
weighted avg,0.9352,0.9298,0.9316,12000.0000


## Confusion matrix, `testbed`

`full`'s confusion matrix is nearly the identity (6 misclassified rows out of 12,000) and isn't informative to look at directly — see `runs/confusion_full.png`. `testbed`'s is the one worth reading.

In [3]:
from sklearn.metrics import confusion_matrix

r = results['testbed']
class_names = list(r['label_encoder'].classes_)
pred = r['proba'].argmax(axis=1)
cm = pd.DataFrame(confusion_matrix(r['y_test'], pred), index=class_names, columns=class_names)
cm.index.name = 'actual'
cm.columns.name = 'predicted'
cm

predicted,Fraudulent Return,Legitimate,Policy Abuser,Wardrobing
actual,,,,
Fraudulent Return,1076,13,10,13
Legitimate,8,7852,274,211
Policy Abuser,1,129,1216,68
Wardrobing,0,52,64,1013


Legitimate ↔ Policy Abuser is the largest confusion band (274 + 129 rows) — the per-class SHAP breakdown in `runs/shap_interpretation.md` explains this mechanistically: on `testbed`, the two classes' top SHAP drivers *overlap* instead of pointing at disjoint generator ranges, unlike `full`.